# Model Optimization Workshop: Introduction and Setup

Welcome to the Model Optimization Workshop! In this workshop, you'll learn how to optimize machine learning models to improve performance and reduce costs. This first notebook will guide you through setting up your environment and downloading the necessary models.

# Part 1: Environment Setup

## 1. Install Required Packages

We'll install only the essential packages needed for the notebook instance. The optimization tasks will run on separate instances with their own environments.

In [ ]:
!pip install -q transformers boto3 sagemaker pandas matplotlib seaborn huggingface_hub requests tqdm numpy

## 2. Verify Environment

In [ ]:
import sys
import boto3
import sagemaker
import transformers
import json
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForTokenClassification
from transformers import AutoModelForQuestionAnswering, AutoModelForMaskedLM
import time
import requests
from huggingface_hub import HfApi, hf_hub_url
from tqdm.auto import tqdm

print(f"Python version: {sys.version}")
print(f"Transformers version: {transformers.__version__}")
print(f"Boto3 version: {boto3.__version__}")
print(f"SageMaker SDK version: {sagemaker.__version__}")

# Check if we're running in a SageMaker notebook
try:
    role = sagemaker.get_execution_role()
    print("Running in a SageMaker notebook instance")
except:
    print("Not running in a SageMaker notebook instance")

## 3. Configure Workshop Settings

Update the following variables with the values from your CloudFormation stack outputs. These settings will be stored and available in all other notebooks.

In [ ]:
# Configure workshop settings
# IMPORTANT: Replace these values with your own from the CloudFormation stack outputs
S3_BUCKET = "YOUR_BUCKET_NAME_HERE"  # Example: model-optimization-workshop-123456789012-us-east-1
AWS_REGION = "YOUR_REGION_HERE"      # Example: us-east-1
OPTIMIZATION_INSTANCE_TYPE = "ml.c5.xlarge"  # Default optimization instance type

# Get the SageMaker execution role automatically
import sagemaker
SAGEMAKER_ROLE_ARN = sagemaker.get_execution_role()
print(f"SageMaker execution role: {SAGEMAKER_ROLE_ARN}")

# Verify the settings
print(f"S3 Bucket: {S3_BUCKET}")
print(f"AWS Region: {AWS_REGION}")
print(f"SageMaker Role ARN: {SAGEMAKER_ROLE_ARN}")
print(f"Optimization Instance Type: {OPTIMIZATION_INSTANCE_TYPE}")

# Check if the bucket exists
try:
    import boto3
    s3 = boto3.resource('s3')
    s3.meta.client.head_bucket(Bucket=S3_BUCKET)
    print(f"\n✅ Successfully connected to S3 bucket: {S3_BUCKET}")
except Exception as e:
    print(f"\n❌ Error connecting to S3 bucket: {e}")
    print("Please check your bucket name and ensure it exists.")

# Store variables for use in other notebooks
%store S3_BUCKET
%store AWS_REGION
%store SAGEMAKER_ROLE_ARN
%store OPTIMIZATION_INSTANCE_TYPE

print("\n✅ Settings stored and available for other notebooks")

# Part 3: Model Selection and Download

## 1. Import Dependencies for Model Selection

## 2. Define Models to Download

We'll download several models for different tasks to demonstrate optimization techniques across various model architectures and sizes.

In [ ]:
# Define models to download
models_to_download = {
    "sentiment_analysis": {
        "model_name": "distilbert-base-uncased-finetuned-sst-2-english",
        "task": "sequence-classification",
        "description": "DistilBERT model fine-tuned for sentiment analysis"
    },
    "ner": {
        "model_name": "dbmdz/bert-large-cased-finetuned-conll03-english",
        "task": "token-classification",
        "description": "BERT model fine-tuned for named entity recognition"
    },
    "question_answering": {
        "model_name": "distilbert-base-cased-distilled-squad",
        "task": "question-answering",
        "description": "DistilBERT model fine-tuned for question answering"
    },
    "masked_lm": {
        "model_name": "bert-base-uncased",
        "task": "masked-lm",
        "description": "BERT model for masked language modeling"
    }
}

## 3. Create Function to Download Models Directly to S3

In [ ]:

def download_model_to_s3(model_name, bucket_name, task, prefix=None):
    """Download model from Hugging Face and upload directly to S3 without local storage."""
    if prefix is None:
        prefix = f"models/{model_name.replace('/', '_')}"
    
    print(f"Downloading {model_name} for {task} directly to S3...")
    
    # Initialize Hugging Face API and S3 client
    hf_api = HfApi()
    s3_client = boto3.client('s3')
    
    try:
        # Get list of model files
        model_files = hf_api.list_repo_files(model_name)
        
        # Upload each file to S3
        for file_name in tqdm(model_files):
            # Get file URL
            file_url = hf_hub_url(model_name, filename=file_name)
            
            # Stream file to S3 without saving locally
            response = requests.get(file_url, stream=True)
            if response.status_code == 200:
                s3_key = f"{prefix}/{file_name}"
                # This is the key part - we're using the raw response stream directly
                s3_client.upload_fileobj(response.raw, bucket_name, s3_key)
            else:
                print(f"Failed to download {file_name}: HTTP {response.status_code}")
        
        # Return S3 URI for the model
        s3_uri = f"s3://{bucket_name}/{prefix}"
        print(f"Model uploaded to {s3_uri}")
        
        return {
            "model_name": model_name,
            "task": task,
            "s3_uri": s3_uri,
            "s3_prefix": prefix
        }
    except Exception as e:
        print(f"Error downloading model {model_name}: {e}")
        return None

## 4. Download Models Directly to S3

In [ ]:
# Download models directly to S3
downloaded_models = {}
for key, model_info in models_to_download.items():
    downloaded_models[key] = download_model_to_s3(
        model_info["model_name"], 
        S3_BUCKET, 
        model_info["task"]
    )
    
    # Add a small delay between downloads to avoid rate limiting
    time.sleep(2)

## 5. Save Model Information for Later Use

In [ ]:
# Create model_info.json with the downloaded model information
model_info = {}
for key, model_data in downloaded_models.items():
    if model_data:  # Check if download was successful
        model_info[key] = {
            "model_name": model_data["model_name"],
            "task": model_data["task"],
            "s3_uri": model_data["s3_uri"],
            "s3_prefix": model_data["s3_prefix"]
        }

# Save model information to file
with open('model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)

print(f"Model information saved to model_info.json")
print(f"Downloaded {len(model_info)} models to S3 bucket: {S3_BUCKET}")

## 6. Next Steps

Now that we've set up our environment and downloaded the necessary models, we're ready to move on to the next notebook where we'll establish baseline performance metrics for each model.